# 2.4 Machine Learning

Llegó la hora que estábamos esperando. Ahora sí escribiremos código en Python para hacer Machine Learning.

Pareciera que llevamos mucho tiempo escribiendo código y no hemos hecho nada de Machine Learning, pero la realidad es que estas fases previas de limpieza de datos y feature engineering son necesarias para poder modelar.

La mayoría de la gente piensa que hacer Machine Learning implica estar corriendo nuevos modelos constantemente. La realidad es muy distinta:

> **La gran mayoría del tiempo se va limpiando y organizando los datos con los que queremos trabajar.**

## Importar paquetes

Estamos en un nuevo notebook, entonces tenemos que volver a importar los paquetes que usaremos.

> ⚠️ **Atención:** Es probable que obtengas un error sobre la librería `LGBMClassifier`. Para resolverlo, primero instala la dependencia del sistema:
>
> **Mac:** `brew install libomp`  
> **Ubuntu:** `sudo apt-get install libgomp1`
>
> Y posteriormente con tu entorno activo:
> ```
> pip install xgboost lightgbm
> ```

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score

# Para guardar el modelo
import pickle

## Carga de datos

En el notebook anterior guardamos nuestros datos procesados en el directorio `data/`. Carguemos estos datos a DataFrames.

In [ ]:
df = pd.read_csv('./data/titanic_procesado.csv')
df.head()

## División de datos

Recordemos que estamos por implementar algoritmos de aprendizaje supervisado. Esto implica que necesitamos dividir nuestros datos en dos conjuntos: uno para **entrenamiento** y otro para **pruebas**.

- El **conjunto de entrenamiento** se utiliza para ajustar el modelo — para que "aprenda" las respuestas correctas.
- El **conjunto de pruebas** se usa para evaluar su rendimiento y asegurarnos de que generaliza bien a datos no vistos.

Esta división se conoce como **división entrenamiento-prueba** (train-test split).

### train_test_split

Utilizaremos la función `train_test_split` de Scikit-Learn para realizar esta división.

Primero crearemos:
- **`X`**: DataFrame con todas las variables excepto la variable objetivo `Survived`
- **`y`**: Serie con únicamente la variable objetivo `Survived`

In [ ]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()

Notamos que nuestra `X` ya no contiene `Survived`. Veamos `y`:

In [ ]:
y.head()

Hagamos la división entrenamiento-prueba:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.values  # Convertir a NumPy array
y_train = y_train.values  # Convertir a NumPy array
X_test  = X_test.values   # Convertir a NumPy array
y_test  = y_test.values   # Convertir a NumPy array

print(f"Registros en entrenamiento: {X_train.shape[0]}")
print(f"Registros en prueba:        {X_test.shape[0]}")

## Entrenamiento de Modelos

Ahora entrenaremos **10 modelos** de clasificación distintos y compararemos su rendimiento sobre el mismo conjunto de datos. Esto nos permitirá identificar cuál algoritmo se adapta mejor al problema.

Los modelos que probaremos son:

| Modelo | Tipo |
|--------|------|
| Logistic Regression | Lineal |
| SVC | Máquinas de soporte vectorial |
| Decision Tree | Árbol de decisión |
| K-Nearest Neighbors | Basado en distancia |
| Random Forest | Ensamble (bagging) |
| Gradient Boosting | Ensamble (boosting) |
| AdaBoost | Ensamble (boosting) |
| XGBoost | Ensamble (boosting extremo) |
| LightGBM | Ensamble (boosting ligero) |
| Gaussian Naive Bayes | Probabilístico |

La estrategia es simple: para cada modelo, llamamos `.fit()` con los datos de entrenamiento y luego `.predict()` con los de prueba para calcular el `accuracy_score`.

In [ ]:
modelos = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, random_state=42),
    'SVC':                   SVC(random_state=42),
    'Decision Tree':         DecisionTreeClassifier(random_state=42),
    'K-Nearest Neighbors':   KNeighborsClassifier(),
    'Random Forest':         RandomForestClassifier(random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(random_state=42),
    'AdaBoost':              AdaBoostClassifier(random_state=42),
    'XGBoost':               XGBClassifier(random_state=42, verbosity=0),
    'LightGBM':              LGBMClassifier(random_state=42, verbose=-1),
    'Gaussian Naive Bayes':  GaussianNB(),
}

resultados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultados[nombre] = acc
    print(f"{nombre:<25} Accuracy: {acc:.4f}")

print("\nEntrenamiento completado.")

## Comparación de Resultados

Ordenemos los modelos de mayor a menor accuracy y visualicemos los resultados.

In [ ]:
import matplotlib.pyplot as plt

# Tabla ordenada
df_resultados = pd.DataFrame(
    list(resultados.items()),
    columns=['Modelo', 'Accuracy']
).sort_values('Accuracy', ascending=False).reset_index(drop=True)

df_resultados.index += 1
print(df_resultados.to_string())

# Gráfica
fig, ax = plt.subplots(figsize=(12, 6))
colores = ['gold' if i == 0 else 'steelblue' for i in range(len(df_resultados))]
bars = ax.barh(
    df_resultados['Modelo'][::-1],
    df_resultados['Accuracy'][::-1],
    color=colores[::-1], edgecolor='white'
)
ax.set_xlim(0.7, 1.0)
ax.set_xlabel('Accuracy')
ax.set_title('Comparacion de Modelos — Accuracy en conjunto de prueba')
for bar, acc in zip(bars, df_resultados['Accuracy'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{acc:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

mejor = df_resultados.iloc[0]
print(f"\nMejor modelo: {mejor['Modelo']}  (Accuracy: {mejor['Accuracy']:.4f})")

## Entrenamiento Individual de un Modelo

Para comprender la búsqueda en cuadrícula (Grid Search) más fácilmente, primero veamos cómo se entrena un modelo individualmente.

Entrenemos una regresión logística:

In [ ]:
# Esto ya lo tenemos importado. Lo ponemos nuevamente nada más de referencia
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Creamos el modelo de regresión logística
model = LogisticRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

# Realizamos predicciones con el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluamos el modelo usando precisión
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

## ¿Y los hiperparámetros?

Logramos un **80%** de precisión, pero este modelo usó los valores por defecto. Cada algoritmo tiene **hiperparámetros** — parámetros que no se aprenden del entrenamiento sino que nosotros configuramos antes de entrenar.

Por ejemplo, `LogisticRegression` tiene entre otros:

| Hiperparámetro | Descripción | Default |
|---------------|-------------|---------|
| `C` | Inverso de la regularización (mayor C = menos regularización) | `1.0` |
| `solver` | Algoritmo de optimización (`lbfgs`, `liblinear`, `saga`…) | `'lbfgs'` |
| `max_iter` | Número máximo de iteraciones para converger | `100` |

Podríamos probar distintos valores manualmente, pero eso sería muy tedioso. Para eso existe **Grid Search**.

## Optimización de Hiperparámetros con GridSearchCV

Cuando vemos la lista larga de hiperparámetros, y todos los posibles valores que podemos dar a cada uno de ellos, sin duda nos sentimos algo abrumados. Después de todo, ¿cómo vamos a aprendernos todas estas combinaciones?

**¡Éste es precisamente el problema que GridSearch busca resolver!**

En lugar de probar combinaciones de hiperparámetros uno por uno, implementamos un GridSearch que probará **todas** las diferentes combinaciones de hiperparámetros que especifiquemos.

Por ejemplo, una configuración simple:

```python
'Regresión Logística': {
    'modelo': LogisticRegression(),
    'parametros': {
        'C': [0.1],
        'max_iter': [1000]
    }
}
```

Podemos expandirla para probar muchas más combinaciones:

```python
'Regresión Logística': {
    'modelo': LogisticRegression(),
    'parametros': {
        'C': [0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga'],
        'max_iter': [100, 500, 1000]
    }
}
```

### ¿Qué hace GridSearch por detrás?

Con la configuración de arriba, GridSearch construirá literalmente **un modelo por cada combinación posible**:

```
LogisticRegression(C=0.01, penalty="l1", solver="liblinear", max_iter=100)
LogisticRegression(C=0.01, penalty="l1", solver="liblinear", max_iter=500)
LogisticRegression(C=0.01, penalty="l1", solver="liblinear", max_iter=1000)
...
LogisticRegression(C=100,  penalty="l2", solver="saga",      max_iter=1000)
```

5 valores de `C` × 2 de `penalty` × 2 de `solver` × 3 de `max_iter` = **60 modelos** — todos evaluados con validación cruzada.

Esto realmente representa la ventaja del machine learning moderno. La regresión logística existe desde el siglo pasado, pero lo que sí es nuevo es la capacidad de contar con herramientas que nos permiten probar hipótesis y experimentar con modelos de manera mucho más rápida y eficiente.

### Diccionario de modelos e hiperparámetros

Definimos todos los modelos junto con sus hiperparámetros en un solo diccionario. Esto nos permite iterar sobre todos ellos con un simple ciclo `for`.

In [ ]:
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'SVC': {
        'modelo': SVC(),
        'parametros': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf'],
            'gamma': ['scale', 'auto']
        }
    },
    'Decision Tree': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'max_depth': [None, 5, 10, 20],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    },
    'K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'n_neighbors': [3, 5, 7, 9, 11],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'Random Forest': {
        'modelo': RandomForestClassifier(random_state=42),
        'parametros': {
            'n_estimators': [100, 200],
            'max_depth': [None, 5, 10],
            'min_samples_split': [2, 5],
            'max_features': ['sqrt', 'log2']
        }
    },
    'Gradient Boosting': {
        'modelo': GradientBoostingClassifier(random_state=42),
        'parametros': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [3, 5]
        }
    },
    'AdaBoost': {
        'modelo': AdaBoostClassifier(random_state=42),
        'parametros': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.5, 1.0, 1.5]
        }
    },
    'XGBoost': {
        'modelo': XGBClassifier(random_state=42, verbosity=0),
        'parametros': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5]
        }
    },
    'LightGBM': {
        'modelo': LGBMClassifier(random_state=42, verbose=-1),
        'parametros': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5],
            'num_leaves': [20, 31]
        }
    },
    'Gaussian Naive Bayes': {
        'modelo': GaussianNB(),
        'parametros': {
            'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
        }
    },
}

### Variables auxiliares

Después de definir el diccionario, creamos unas variables auxiliares que nos servirán para almacenar los resultados y encontrar fácilmente el mejor modelo:

In [ ]:
# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision  = 0
mejor_estimador  = None
mejor_modelo     = None
estimadores      = {}

### Ajuste de modelos

El proceso de ajuste lo haremos por cada uno de los elementos de nuestro diccionario utilizando un ciclo `for`. Adentro del ciclo, `GridSearchCV` probará todas las combinaciones de hiperparámetros con validación cruzada de 5 folds:

In [ ]:
# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():

    grid_search = GridSearchCV(
        estimator  = info_modelo['modelo'],
        param_grid = info_modelo['parametros'],
        cv         = 5,
        scoring    = 'accuracy',
        verbose    = 0,
        n_jobs     = -1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)

    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)

    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)

    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo':    nombre,
        'Precision': precision
    })

    estimadores[nombre] = grid_search.best_estimator_

    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo    = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

Una vez creado el objeto `grid_search`, procedemos a ajustar el modelo y posteriormente hacemos predicciones utilizando `X_test`. Los resultados se almacenan en `y_pred`.

Para medir la precisión, comparamos `y_pred` con `y_test` para "ver a cuántos le atinó el modelo", y guardamos todo en `puntajes_modelos`.

El bloque `if` dentro del ciclo mantiene actualizado el mejor modelo encontrado hasta ese punto. Al terminar el `for`, `mejor_estimador` contiene el modelo ganador.

Ahora escribiremos código para mostrar los resultados:

In [ ]:
# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precision', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2).to_string())

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo:    {mejor_modelo}")
print(f"Precision: {mejor_precision:.2f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

colores = ['gold' if i == 0 else 'mediumseagreen' for i in range(len(metricas))]
bars = ax.barh(
    metricas['Modelo'][::-1],
    metricas['Precision'][::-1],
    color=colores[::-1], edgecolor='white'
)
ax.set_xlim(0.65, 0.95)
ax.set_xlabel('Precision')
ax.set_title('Ranking de modelos — GridSearchCV')
for bar, acc in zip(bars, metricas['Precision'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{acc:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Guardar el modelo

Este último paso es necesario para la siguiente lección en la que pondremos nuestro modelo en producción. Usaremos un paquete de Python llamado **Pickle**, el cual nos permite serializar (guardar) objetos de Python en un archivo para luego poder cargarlos y utilizarlos en diferentes entornos, como una API o una aplicación web.

Pickle es particularmente útil cuando queremos guardar modelos entrenados o cualquier otro objeto complejo de Python. Al guardar el modelo con Pickle, nos aseguramos de que todas las transformaciones de datos y el modelo en sí se conserven tal como fueron entrenados, permitiéndonos hacer predicciones consistentes con nuevos datos en producción sin necesidad de volver a aplicar las mismas transformaciones manualmente.

## Predicción con nuevos datos

En el contexto de nuestro proyecto, podríamos ya usar nuestro modelo para alimentarle nueva información de pasajeros y predecir si sobrevivió o no.

En nuestro proceso de entrenamiento, guardamos el mejor estimador en la variable `mejor_estimador`. Si vemos el contenido de esta variable en Jupyter, veremos el objeto del modelo con sus hiperparámetros ajustados:

In [ ]:
mejor_estimador

Para predecir nuevos valores usaremos el método `predict`, que recibe un numpy array como argumento. Este array debe tener exactamente las mismas dimensiones que `X_train` y `X_test`.

Obtengamos los primeros datos de `X_train` y de `y_train`:

In [ ]:
X_train[0]

In [ ]:
y_train[0]

Ahora creemos un nuevo numpy array con los datos que vemos en `X_train[0]` y corramos `predict`:

In [ ]:
nuevos_datos = np.array([0, 1, 0.6159084, 0, 0, 0.55547282, 1]).reshape(1, -1)

mejor_estimador.predict(nuevos_datos)

Nuestro modelo predice que este pasajero **no sobrevivió** (`array([0])`), lo que coincide con `y_train[0]`.

Si no has importado `pickle`, hazlo ahora:

In [ ]:
import pickle

with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print("Modelo guardado exitosamente como 'modelo.pkl'")